# CSR Attention Sorting Locality Benchmark

This notebook measures how query/key memory order affects CSR attention locality and CUDA forward/backward runtime. Geometry construction, graph remapping, CSR building, and locality metrics use PyTorch CUDA tensors where possible. The CSR attention backend is imported directly; no JIT build is performed here.

In [1]:
from pathlib import Path
import os
import time

import pandas as pd
import torch
import trimesh

ROOT = Path('/mnt/nvmefs/Projects/SymTRELLIS')
DEVICE = torch.device('cuda')
assert torch.cuda.is_available(), 'CUDA is required for this notebook'

from o_voxel.convert import intersect_occ
from symtrellis.geometry import lattice_ball_offsets, radius_nbr_edges
from symtrellis.mapper.attention.csr_attn import sparse_csr_attn_backend

G = 256
RADIUS = 0.4
MESH_PATH = ROOT / 'notebooks/test.glb'
MESH_AABB = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]]
NEIGHBOR_RADIUS = 2.0
ROTATION_SEED = 20260628
RANDOM_ORDER_SEED = 20260629
FEATURE_SEED = 20260630
H = 4
D = 32
BARYCENTRIC_ITERS = 5
BENCH_REPEATS = 10
VOXEL_Z_CHUNK = 16

torch.manual_seed(FEATURE_SEED)
torch.cuda.manual_seed_all(FEATURE_SEED)

print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('device:', torch.cuda.get_device_name(DEVICE))
print('G:', G, 'RADIUS:', RADIUS, 'NEIGHBOR_RADIUS:', NEIGHBOR_RADIUS)
print('mesh:', MESH_PATH)
print('H:', H, 'D:', D, 'BARYCENTRIC_ITERS:', BARYCENTRIC_ITERS, 'BENCH_REPEATS:', BENCH_REPEATS)

torch: 2.6.0+cu124 cuda: 12.4
device: NVIDIA GeForce RTX 4090
G: 256 RADIUS: 0.4 NEIGHBOR_RADIUS: 2.0
mesh: /mnt/nvmefs/Projects/SymTRELLIS/notebooks/test.glb
H: 4 D: 32 BARYCENTRIC_ITERS: 5 BENCH_REPEATS: 10


## GPU Helpers

Large tensor operations stay on CUDA. CPU transfer is limited to final scalar table rows.

In [2]:
def cuda_timed(fn):
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    result = fn()
    end.record()
    torch.cuda.synchronize()
    return result, float(start.elapsed_time(end))


def scalar(x):
    return float(x.detach().item())


def qtile(x, q):
    if x.numel() == 0:
        return float('nan')
    return scalar(torch.quantile(x.float(), q))


def rank_from_scores(scores):
    order = torch.argsort(scores)
    rank = torch.empty_like(order, dtype=torch.long)
    rank[order] = torch.arange(order.numel(), device=scores.device, dtype=torch.long)
    return rank, order


def grid_to_pos(grid):
    return (grid.to(torch.float32) + 0.5) / float(G) - 0.5


def random_rotation(seed, device):
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)
    q = torch.randn((4,), device=device, dtype=torch.float32, generator=gen)
    q = q / q.norm()
    w, x, y, z = q.unbind()
    return torch.stack([
        torch.stack([1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)]),
        torch.stack([2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)]),
        torch.stack([2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]),
    ])


def linear_code(coords):
    c = coords.to(torch.long)
    return c[:, 0] + G * c[:, 1] + (G * G) * c[:, 2]


def morton3d_code(coords):
    c = coords.to(torch.long)
    x, y, z = c[:, 0], c[:, 1], c[:, 2]
    code = torch.zeros_like(x)
    for b in range(8):
        code |= ((x >> b) & 1) << (3 * b)
        code |= ((y >> b) & 1) << (3 * b + 1)
        code |= ((z >> b) & 1) << (3 * b + 2)
    return code


def coords_to_q_grid(pos_world):
    q_grid = torch.floor((pos_world + 0.5) * float(G)).to(torch.long)
    q_grid = q_grid.clamp_(0, G - 1)
    return q_grid.to(torch.int32)

## Voxelize Mesh Q/K and Build Radius Graph

The analytic sphere voxelizer is kept as a simple reference path. The benchmark input below loads `notebooks/test.glb`, normalizes it around the bbox center into a radius-0.5 ball, voxelizes Q once, then voxelizes K/V after a random rotation.

In [3]:
@torch.no_grad()
def voxelize_sphere_surface():
    xs = torch.arange(G, device=DEVICE, dtype=torch.int32)
    ys = torch.arange(G, device=DEVICE, dtype=torch.int32)
    chunks = []
    r2 = float(RADIUS * RADIUS)
    for z0 in range(0, G, VOXEL_Z_CHUNK):
        z1 = min(z0 + VOXEL_Z_CHUNK, G)
        zs = torch.arange(z0, z1, device=DEVICE, dtype=torch.int32)
        xx, yy, zz = torch.meshgrid(xs, ys, zs, indexing='ij')
        grid = torch.stack([xx, yy, zz], dim=-1)
        lo = grid.to(torch.float32) / float(G) - 0.5
        hi = (grid.to(torch.float32) + 1.0) / float(G) - 0.5
        dmin_axis = torch.where(lo > 0, lo, torch.where(hi < 0, -hi, torch.zeros_like(lo)))
        dmax_axis = torch.maximum(lo.abs(), hi.abs())
        dmin2 = (dmin_axis * dmin_axis).sum(dim=-1)
        dmax2 = (dmax_axis * dmax_axis).sum(dim=-1)
        mask = (dmin2 <= r2) & (r2 <= dmax2)
        if mask.any():
            chunks.append(grid[mask].reshape(-1, 3))
    return torch.cat(chunks, dim=0).contiguous()


def load_trimesh_mesh(path):
    loaded = trimesh.load(path, force='scene')
    if isinstance(loaded, trimesh.Scene):
        mesh = loaded.to_geometry()
        if not isinstance(mesh, trimesh.Trimesh) or len(mesh.vertices) == 0 or len(mesh.faces) == 0:
            raise ValueError(f'no triangle mesh found in {path}')
    elif isinstance(loaded, trimesh.Trimesh):
        mesh = loaded
    else:
        raise TypeError(f'unsupported mesh object: {type(loaded)}')
    mesh.remove_unreferenced_vertices()
    return mesh


@torch.no_grad()
def load_normalized_mesh(path):
    mesh = load_trimesh_mesh(path)
    vertices = torch.as_tensor(mesh.vertices, device=DEVICE, dtype=torch.float32).contiguous()
    faces = torch.as_tensor(mesh.faces, device=DEVICE, dtype=torch.long).contiguous()
    raw_min = vertices.min(dim=0).values
    raw_max = vertices.max(dim=0).values
    center = (raw_min + raw_max) * 0.5
    vertices = vertices - center
    raw_radius = vertices.norm(dim=1).max()
    scale = 0.5 / raw_radius
    vertices = (vertices * scale).contiguous()
    info = {
        'mesh_vertices': int(vertices.shape[0]),
        'mesh_faces': int(faces.shape[0]),
        'mesh_scale': scalar(scale),
        'mesh_raw_radius': scalar(raw_radius),
        'mesh_center_x': scalar(center[0]),
        'mesh_center_y': scalar(center[1]),
        'mesh_center_z': scalar(center[2]),
    }
    return vertices, faces, info


@torch.no_grad()
def voxelize_mesh_pair():
    vertices, faces, info = load_normalized_mesh(MESH_PATH)
    rot = random_rotation(ROTATION_SEED, DEVICE)

    q_grid = intersect_occ(vertices, faces, grid_size=G, aabb=MESH_AABB).to(torch.int32).contiguous()
    vertices_rot = (vertices @ rot).contiguous()
    k_grid_rot = intersect_occ(vertices_rot, faces, grid_size=G, aabb=MESH_AABB).to(torch.int32).contiguous()

    q_pos_world = grid_to_pos(q_grid)
    k_pos_rot = grid_to_pos(k_grid_rot)
    k_pos_world = k_pos_rot @ rot.T
    k_grid_q = coords_to_q_grid(k_pos_world)
    q_pos_rot = q_pos_world @ rot
    q_pos_kgrid = (q_pos_rot + 0.5) * float(G) - 0.5

    return {
        'q_grid': q_grid,
        'k_grid_rot': k_grid_rot,
        'q_pos_world': q_pos_world,
        'k_pos_world': k_pos_world,
        'k_grid_q': k_grid_q,
        'q_pos_kgrid': q_pos_kgrid,
        'rot': rot,
        'mesh_info': info,
    }


mesh_data, voxelize_ms = cuda_timed(voxelize_mesh_pair)
q_grid = mesh_data['q_grid']
k_grid_rot = mesh_data['k_grid_rot']
q_pos_world = mesh_data['q_pos_world']
k_pos_world = mesh_data['k_pos_world']
k_grid_q = mesh_data['k_grid_q']
q_pos_kgrid = mesh_data['q_pos_kgrid']
rot = mesh_data['rot']
mesh_info = mesh_data['mesh_info']

Nq = int(q_grid.shape[0])
Nk = int(k_grid_rot.shape[0])
print('voxelize_ms:', voxelize_ms)
print('Nq:', Nq, 'Nk:', Nk)
print('mesh vertices/faces:', mesh_info['mesh_vertices'], mesh_info['mesh_faces'])
print('rotation det:', scalar(torch.linalg.det(rot)))
print('q_grid dtype/device:', q_grid.dtype, q_grid.device)
print('k_grid_q range:', int(k_grid_q.min().item()), int(k_grid_q.max().item()))

voxelize_ms: 198.5250244140625
Nq: 487183 Nk: 510916
mesh vertices/faces: 268018 280333
rotation det: 0.9999998211860657
q_grid dtype/device: torch.int32 cuda:0
k_grid_q range: 23 232


In [4]:
@torch.no_grad()
def build_radius_graph():
    query_bid = torch.zeros((Nq,), device=DEVICE, dtype=torch.int32)
    key_bid = torch.zeros((Nk,), device=DEVICE, dtype=torch.int32)
    nbr_offsets = lattice_ball_offsets(NEIGHBOR_RADIUS, device=DEVICE).to(torch.int32)
    coord_min = k_grid_rot.min(dim=0).values.to(torch.int32)
    return radius_nbr_edges(
        query_pos=q_pos_kgrid.contiguous(),
        query_bid=query_bid,
        key_coords=k_grid_rot.contiguous(),
        key_bid=key_bid,
        radius=float(NEIGHBOR_RADIUS),
        nbr_offsets=nbr_offsets,
        coord_min=coord_min,
    )


(e_qids, e_kids), graph_ms = cuda_timed(build_radius_graph)
E = int(e_qids.numel())
deg_q = torch.bincount(e_qids, minlength=Nq)
deg_k = torch.bincount(e_kids, minlength=Nk)
nonempty_q = deg_q > 0
graph_summary = pd.DataFrame([{
    'source': 'test.glb',
    'G': G,
    'mesh_path': str(MESH_PATH),
    **mesh_info,
    'neighbor_radius': NEIGHBOR_RADIUS,
    'Nq': Nq,
    'Nk': Nk,
    'E': E,
    'degree_min': int(deg_q.min().item()),
    'degree_mean': scalar(deg_q.float().mean()),
    'degree_median': qtile(deg_q.float(), 0.5),
    'degree_p95': qtile(deg_q.float(), 0.95),
    'degree_max': int(deg_q.max().item()),
    'empty_query_rows': int((deg_q == 0).sum().item()),
    'isolated_keys': int((deg_k == 0).sum().item()),
    'voxelize_ms': voxelize_ms,
    'graph_ms': graph_ms,
    'rotation_seed': ROTATION_SEED,
}])
display(graph_summary)

,source,G,mesh_path,mesh_vertices,mesh_faces,mesh_scale,mesh_raw_radius,mesh_center_x,mesh_center_y,mesh_center_z,...,degree_min,degree_mean,degree_median,degree_p95,degree_max,empty_query_rows,isolated_keys,voxelize_ms,graph_ms,rotation_seed
0,test.glb,256,/mnt/nvmefs/Projects/SymTRELLIS/notebooks/test...,268018,280333,0.823705,0.607013,0.00389,-0.000544,0.002542,...,4,26.996956,29.0,36.0,40,0,0,198.525024,39.46365,20260628


## Sorting Methods

All coordinate-based orders use Q/world grid coordinates: `q_grid` for Q and `k_grid_q` for K.

In [5]:
def average_by_index(index, values, size, counts=None):
    out = torch.zeros((size,), device=values.device, dtype=torch.float32)
    out.index_add_(0, index, values.float())
    if counts is None:
        counts = torch.bincount(index, minlength=size)
    return out / counts.clamp_min(1).float()


def method_original_order():
    return torch.arange(Nq, device=DEVICE, dtype=torch.long), torch.arange(Nk, device=DEVICE, dtype=torch.long)


def method_random_order():
    gen = torch.Generator(device=DEVICE)
    gen.manual_seed(RANDOM_ORDER_SEED)
    perm_q = torch.randperm(Nq, device=DEVICE, generator=gen)
    perm_k = torch.randperm(Nk, device=DEVICE, generator=gen)
    pi_q = torch.empty_like(perm_q)
    pi_k = torch.empty_like(perm_k)
    pi_q[perm_q] = torch.arange(Nq, device=DEVICE, dtype=torch.long)
    pi_k[perm_k] = torch.arange(Nk, device=DEVICE, dtype=torch.long)
    return pi_q, pi_k


def method_linear_grid_order():
    pi_q, _ = rank_from_scores(linear_code(q_grid))
    pi_k, _ = rank_from_scores(linear_code(k_grid_q))
    return pi_q, pi_k


def method_morton_order():
    pi_q, _ = rank_from_scores(morton3d_code(q_grid))
    pi_k, _ = rank_from_scores(morton3d_code(k_grid_q))
    return pi_q, pi_k


def method_key_morton_query_neighbor_center():
    pi_k, _ = rank_from_scores(morton3d_code(k_grid_q))
    center = average_by_index(e_qids, pi_k[e_kids].float(), Nq, deg_q)
    empty = deg_q == 0
    if empty.any():
        center[empty] = float(Nk) + torch.arange(int(empty.sum().item()), device=DEVICE, dtype=torch.float32)
    pi_q, _ = rank_from_scores(center)
    return pi_q, pi_k


def method_bipartite_barycentric_order():
    pi_k, _ = rank_from_scores(morton3d_code(k_grid_q))
    p_k = pi_k.float()
    for _ in range(BARYCENTRIC_ITERS):
        p_q = average_by_index(e_qids, p_k[e_kids], Nq, deg_q)
        empty_q = deg_q == 0
        if empty_q.any():
            p_q[empty_q] = float(Nk) + torch.arange(int(empty_q.sum().item()), device=DEVICE, dtype=torch.float32)
        pi_q, _ = rank_from_scores(p_q)
        p_q = pi_q.float()

        p_k = average_by_index(e_kids, p_q[e_qids], Nk, deg_k)
        empty_k = deg_k == 0
        if empty_k.any():
            p_k[empty_k] = float(Nq) + torch.arange(int(empty_k.sum().item()), device=DEVICE, dtype=torch.float32)
        pi_k, _ = rank_from_scores(p_k)
        p_k = pi_k.float()
    return pi_q, pi_k


SORT_METHODS = [
    ('original_order', method_original_order),
    ('random_order', method_random_order),
    ('linear_grid_order', method_linear_grid_order),
    ('morton_order', method_morton_order),
    ('key_morton_query_neighbor_center', method_key_morton_query_neighbor_center),
    ('bipartite_barycentric_order', method_bipartite_barycentric_order),
]

## CSR Build and Locality Metrics

In [6]:
@torch.no_grad()
def build_csr(pi_q, pi_k):
    q_new = pi_q[e_qids]
    k_new = pi_k[e_kids]
    edge_key = q_new * int(Nk) + k_new
    order = torch.argsort(edge_key)
    q_sorted = q_new[order]
    col = k_new[order].to(torch.int32).contiguous()
    counts = torch.bincount(q_sorted, minlength=Nq)
    rowptr64 = torch.empty((Nq + 1,), device=DEVICE, dtype=torch.long)
    rowptr64[0] = 0
    rowptr64[1:] = torch.cumsum(counts, dim=0)
    assert int(rowptr64[-1].item()) == E
    assert int(rowptr64[-1].item()) < 2**31
    return rowptr64.to(torch.int32).contiguous(), col


@torch.no_grad()
def locality_metrics(rowptr, col):
    rowptr64 = rowptr.to(torch.long)
    col64 = col.to(torch.long)
    deg = rowptr64[1:] - rowptr64[:-1]
    nonempty = deg > 0

    starts = rowptr64[:-1][nonempty]
    ends = rowptr64[1:][nonempty]
    first = col64[starts]
    last = col64[ends - 1]
    span = (last - first).float()

    if E > 1:
        row_ids = torch.repeat_interleave(torch.arange(Nq, device=DEVICE, dtype=torch.long), deg, output_size=E)
        same_row = row_ids[1:] == row_ids[:-1]
        intra_gap = (col64[1:] - col64[:-1]).float()[same_row]
        intra_rows = row_ids[:-1][same_row]
        gap_sum = torch.zeros((Nq,), device=DEVICE, dtype=torch.float32)
        gap_sum.index_add_(0, intra_rows, intra_gap)
        gap_count = (deg - 1).clamp_min(0)
        valid_gap = gap_count > 0
        row_gap = gap_sum[valid_gap] / gap_count[valid_gap].float()
        csr_col_gap = (col64[1:] - col64[:-1]).abs().float()
        center_sum = torch.zeros((Nq,), device=DEVICE, dtype=torch.float32)
        center_sum.index_add_(0, row_ids, col64.float())
    else:
        row_gap = torch.empty((0,), device=DEVICE, dtype=torch.float32)
        csr_col_gap = torch.empty((0,), device=DEVICE, dtype=torch.float32)
        center_sum = torch.zeros((Nq,), device=DEVICE, dtype=torch.float32)

    centers = center_sum[nonempty] / deg[nonempty].float()
    qjump = (centers[1:] - centers[:-1]).abs() if centers.numel() > 1 else torch.empty((0,), device=DEVICE)

    return {
        'row_span_mean': scalar(span.mean()),
        'row_span_median': qtile(span, 0.5),
        'row_span_p95': qtile(span, 0.95),
        'row_gap_mean': scalar(row_gap.mean()) if row_gap.numel() else float('nan'),
        'row_gap_median': qtile(row_gap, 0.5),
        'row_gap_p95': qtile(row_gap, 0.95),
        'query_center_jump_mean': scalar(qjump.mean()) if qjump.numel() else float('nan'),
        'query_center_jump_p95': qtile(qjump, 0.95),
        'csr_col_gap_mean': scalar(csr_col_gap.mean()) if csr_col_gap.numel() else float('nan'),
        'csr_col_gap_p95': qtile(csr_col_gap, 0.95),
    }


method_data = {}
preprocess_rows = []
metric_rows = []

for method_name, method_fn in SORT_METHODS:
    print('preprocess', method_name)
    (pi_q, pi_k), sort_ms = cuda_timed(method_fn)
    (rowptr, col), csr_ms = cuda_timed(lambda pi_q=pi_q, pi_k=pi_k: build_csr(pi_q, pi_k))
    metrics, metric_ms = cuda_timed(lambda rowptr=rowptr, col=col: locality_metrics(rowptr, col))
    method_data[method_name] = {'pi_q': pi_q, 'pi_k': pi_k, 'rowptr': rowptr, 'col': col}
    preprocess_rows.append({
        'method': method_name,
        'sort_order_time_ms': sort_ms,
        'csr_build_time_ms': csr_ms,
        'metric_time_ms': metric_ms,
        'total_preprocess_time_ms': sort_ms + csr_ms,
    })
    metric_rows.append({'method': method_name, **metrics})

preprocess_df = pd.DataFrame(preprocess_rows)
metrics_df = pd.DataFrame(metric_rows)
display(preprocess_df)
display(metrics_df)

preprocess original_order
preprocess random_order
preprocess linear_grid_order
preprocess morton_order
preprocess key_morton_query_neighbor_center
preprocess bipartite_barycentric_order


,method,sort_order_time_ms,csr_build_time_ms,metric_time_ms,total_preprocess_time_ms
0,original_order,0.318240,14.838784,9.664832,15.157024
1,random_order,3.078336,9.163616,9.450848,12.241952
2,linear_grid_order,0.715520,8.909856,8.846496,9.625376
3,morton_order,9.577824,9.116672,8.053408,18.694495
4,key_morton_query_neighbor_center,2.311296,9.414752,9.054880,11.726048
5,bipartite_barycentric_order,7.910464,9.057184,8.182784,16.967648


,method,row_span_mean,row_span_median,row_span_p95,row_gap_mean,row_gap_median,row_gap_p95,query_center_jump_mean,query_center_jump_p95,csr_col_gap_mean,csr_col_gap_p95
0,original_order,88155.398438,33976.0,325749.68750,3604.909912,1297.742798,13668.349609,23192.378906,97276.976562,6636.308105,16826.0
1,random_order,471445.593750,478778.0,504240.00000,19702.052734,17133.679688,31645.609375,33330.808594,82619.296875,34925.804688,81380.0
2,linear_grid_order,10547.749023,10801.0,15830.00000,426.380798,406.096771,732.103943,351.580231,1163.034668,781.363953,3762.0
3,morton_order,23152.796875,1451.0,136106.90625,881.724182,55.352940,4776.857422,3787.230713,30312.421875,1715.176514,809.0
4,key_morton_query_neighbor_center,23152.794922,1451.0,136106.90625,881.724243,55.352940,4776.857422,1.048588,3.218750,1715.176514,1007.0
5,bipartite_barycentric_order,17737.337891,5952.0,63970.00000,668.043274,232.449997,2435.796143,1763.391846,5471.699219,1314.026611,4317.0


## Float32 CSR Attention Runtime and Memory

In [7]:
gen = torch.Generator(device=DEVICE)
gen.manual_seed(FEATURE_SEED)
q_base = (torch.randn((Nq, H, D), device=DEVICE, dtype=torch.float32, generator=gen) * 0.2).contiguous()
k_base = (torch.randn((Nk, H, D), device=DEVICE, dtype=torch.float32, generator=gen) * 0.2).contiguous()
v_base = torch.randn((Nk, H, D), device=DEVICE, dtype=torch.float32, generator=gen).contiguous()
grad_out_base = torch.randn((Nq, H, D), device=DEVICE, dtype=torch.float32, generator=gen).contiguous()


@torch.no_grad()
def reorder_features(pi_q, pi_k):
    q = torch.empty_like(q_base)
    k = torch.empty_like(k_base)
    v = torch.empty_like(v_base)
    go = torch.empty_like(grad_out_base)
    q[pi_q] = q_base
    k[pi_k] = k_base
    v[pi_k] = v_base
    go[pi_q] = grad_out_base
    return q.contiguous(), k.contiguous(), v.contiguous(), go.contiguous()


def one_attention_run(q_sorted, k_sorted, v_sorted, grad_out_sorted, rowptr, col, measured=True):
    q = q_sorted.detach().clone().requires_grad_(True)
    k = k_sorted.detach().clone().requires_grad_(True)
    v = v_sorted.detach().clone().requires_grad_(True)
    torch.cuda.synchronize()
    if measured:
        torch.cuda.reset_peak_memory_stats()
    base_alloc = torch.cuda.memory_allocated()
    base_reserved = torch.cuda.memory_reserved()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    out = sparse_csr_attn_backend(q, k, v, rowptr, col)
    loss = (out * grad_out_sorted).sum()
    loss.backward()
    end.record()
    torch.cuda.synchronize()
    record = {
        'ms': float(start.elapsed_time(end)),
        'alloc_delta_mb': (torch.cuda.memory_allocated() - base_alloc) / 1024**2,
        'reserved_delta_mb': (torch.cuda.memory_reserved() - base_reserved) / 1024**2,
        'peak_alloc_delta_mb': (torch.cuda.max_memory_allocated() - base_alloc) / 1024**2,
        'peak_reserved_delta_mb': (torch.cuda.max_memory_reserved() - base_reserved) / 1024**2,
    }
    del q, k, v, out, loss
    torch.cuda.synchronize()
    return record


bench_rows = []
for method_name, data in method_data.items():
    print('attention bench', method_name)
    (q_sorted, k_sorted, v_sorted, grad_out_sorted), reorder_ms = cuda_timed(
        lambda data=data: reorder_features(data['pi_q'], data['pi_k'])
    )
    _ = one_attention_run(q_sorted, k_sorted, v_sorted, grad_out_sorted, data['rowptr'], data['col'], measured=False)
    records = [
        one_attention_run(q_sorted, k_sorted, v_sorted, grad_out_sorted, data['rowptr'], data['col'], measured=True)
        for _ in range(BENCH_REPEATS)
    ]
    bench_rows.append({
        'method': method_name,
        'feature_reorder_time_ms': reorder_ms,
        'runtime_ms_mean': sum(r['ms'] for r in records) / len(records),
        'runtime_ms_min': min(r['ms'] for r in records),
        'alloc_delta_mb_mean': sum(r['alloc_delta_mb'] for r in records) / len(records),
        'reserved_delta_mb_mean': sum(r['reserved_delta_mb'] for r in records) / len(records),
        'peak_alloc_delta_mb_mean': sum(r['peak_alloc_delta_mb'] for r in records) / len(records),
        'peak_reserved_delta_mb_mean': sum(r['peak_reserved_delta_mb'] for r in records) / len(records),
        'repeats': BENCH_REPEATS,
    })
    del q_sorted, k_sorted, v_sorted, grad_out_sorted
    torch.cuda.synchronize()

bench_df = pd.DataFrame(bench_rows)
display(bench_df)

attention bench original_order
attention bench random_order
attention bench linear_grid_order
attention bench morton_order
attention bench key_morton_query_neighbor_center
attention bench bipartite_barycentric_order


,method,feature_reorder_time_ms,runtime_ms_mean,runtime_ms_min,alloc_delta_mb_mean,reserved_delta_mb_mean,peak_alloc_delta_mb_mean,peak_reserved_delta_mb_mean,repeats
0,original_order,3.316736,20.893187,20.124672,976.000488,0.0,1221.435059,0.0,10
1,random_order,3.503104,79.468044,76.492798,976.000488,0.0,1221.435059,0.0,10
2,linear_grid_order,3.500768,20.144128,19.239937,976.000488,0.0,1221.435059,0.0,10
3,morton_order,3.462240,20.474237,19.316032,976.000488,0.0,1221.435059,0.0,10
4,key_morton_query_neighbor_center,2.811904,20.354918,19.937344,976.000488,0.0,1221.435059,0.0,10
5,bipartite_barycentric_order,3.659040,20.190426,19.880960,976.000488,0.0,1221.435059,0.0,10


## Combined Summary

In [ ]:
summary = preprocess_df.merge(metrics_df, on='method').merge(bench_df, on='method')
display(summary.sort_values('runtime_ms_mean')[[
    'method',
    'runtime_ms_mean', 'runtime_ms_min',
    'row_span_mean', 'row_span_median', 'row_span_p95',
    'row_gap_mean', 'row_gap_median', 'row_gap_p95',
    'query_center_jump_mean', 'query_center_jump_p95',
    'csr_col_gap_mean', 'csr_col_gap_p95',
    'sort_order_time_ms', 'csr_build_time_ms', 'metric_time_ms', 'feature_reorder_time_ms',
    'total_preprocess_time_ms',
    'peak_alloc_delta_mb_mean', 'peak_reserved_delta_mb_mean',
]])

display(summary.sort_values('row_span_mean')[[
    'method', 'row_span_mean', 'row_gap_mean', 'query_center_jump_mean',
    'csr_col_gap_mean', 'runtime_ms_mean', 'runtime_ms_min',
    'total_preprocess_time_ms', 'feature_reorder_time_ms',
]])

,method,runtime_ms_mean,runtime_ms_min,row_span_mean,row_span_median,row_span_p95,row_gap_mean,row_gap_median,row_gap_p95,query_center_jump_mean,query_center_jump_p95,csr_col_gap_mean,csr_col_gap_p95,sort_order_time_ms,csr_build_time_ms,metric_time_ms,feature_reorder_time_ms,total_preprocess_time_ms,peak_alloc_delta_mb_mean,peak_reserved_delta_mb_mean
2,linear_grid_order,20.144128,19.239937,10547.749023,10801.0,15830.00000,426.380798,406.096771,732.103943,351.580231,1163.034668,781.363953,3762.0,0.715520,8.909856,8.846496,3.500768,9.625376,1221.435059,0.0
5,bipartite_barycentric_order,20.190426,19.880960,17737.337891,5952.0,63970.00000,668.043274,232.449997,2435.796143,1763.391846,5471.699219,1314.026611,4317.0,7.910464,9.057184,8.182784,3.659040,16.967648,1221.435059,0.0
4,key_morton_query_neighbor_center,20.354918,19.937344,23152.794922,1451.0,136106.90625,881.724243,55.352940,4776.857422,1.048588,3.218750,1715.176514,1007.0,2.311296,9.414752,9.054880,2.811904,11.726048,1221.435059,0.0
3,morton_order,20.474237,19.316032,23152.796875,1451.0,136106.90625,881.724182,55.352940,4776.857422,3787.230713,30312.421875,1715.176514,809.0,9.577824,9.116672,8.053408,3.462240,18.694495,1221.435059,0.0
0,original_order,20.893187,20.124672,88155.398438,33976.0,325749.68750,3604.909912,1297.742798,13668.349609,23192.378906,97276.976562,6636.308105,16826.0,0.318240,14.838784,9.664832,3.316736,15.157024,1221.435059,0.0
1,random_order,79.468044,76.492798,471445.593750,478778.0,504240.00000,19702.052734,17133.679688,31645.609375,33330.808594,82619.296875,34925.804688,81380.0,3.078336,9.163616,9.450848,3.503104,12.241952,1221.435059,0.0


,method,row_span_mean,row_gap_mean,query_center_jump_mean,csr_col_gap_mean,runtime_ms_mean,runtime_ms_min,total_preprocess_time_ms,feature_reorder_time_ms
2,linear_grid_order,10547.749023,426.380798,351.580231,781.363953,20.144128,19.239937,9.625376,3.500768
5,bipartite_barycentric_order,17737.337891,668.043274,1763.391846,1314.026611,20.190426,19.880960,16.967648,3.659040
4,key_morton_query_neighbor_center,23152.794922,881.724243,1.048588,1715.176514,20.354918,19.937344,11.726048,2.811904
3,morton_order,23152.796875,881.724182,3787.230713,1715.176514,20.474237,19.316032,18.694495,3.462240
0,original_order,88155.398438,3604.909912,23192.378906,6636.308105,20.893187,20.124672,15.157024,3.316736
1,random_order,471445.593750,19702.052734,33330.808594,34925.804688,79.468044,76.492798,12.241952,3.503104


: 